# BikroyLens — Fair Price Model (Week 3-4)

Trains an XGBoost regressor to predict a phone's fair price from brand, model, storage, condition, city, photo count, and seller type. Then computes a fair-price range (±10%) and a 0-100 deal score for every listing.

Run each cell in order (Shift+Enter). Nothing here is automated — you're driving.

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import psycopg2
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_absolute_percentage_error
import xgboost as xgb

load_dotenv()
DATABASE_URL = os.environ["DATABASE_URL"]

## Step 1 — Pull data from Postgres

`listings` is append-only (the same phone gets re-scraped every day it's still up), so this query deduplicates to **one row per unique listing `url`**, keeping its most recent sighting — the current asking price is what a 'fair price now' model should learn from, not every historical repeat.

In [ ]:
conn = psycopg2.connect(DATABASE_URL)

query = """
    SELECT DISTINCT ON (l.url)
        l.url, l.price, l.location, l.photo_count, l.seller_type,
        n.brand, n.model, n.storage, n.condition_clean
    FROM listings l
    JOIN phones_normalized n ON n.listing_id = l.id
    ORDER BY l.url, l.scraped_date DESC
"""

df = pd.read_sql(query, conn)
conn.close()

print("rows:", len(df))
df.head()

## Missing values check

Before doing anything else, see exactly what's missing and where.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct}).sort_values("missing_count", ascending=False)

## Step 2 — Feature engineering

- `storage`: parse the `"256GB"` text into a plain number (256)
- `city`: bucket `location` (e.g. `"Dhaka, Uttara"`, 144 distinct values) down to just the city (3 values) — the full area-level detail is too sparse to be useful
- `ram` is deliberately **not** used — only ~0.8% of listings have it, essentially all null
- Rows missing `price` (the target) or `brand`/`storage` are dropped — nothing to learn from those

In [ ]:
def parse_storage(val):
    if pd.isna(val):
        return None
    m = re.search(r"(\d+)", str(val))
    return int(m.group(1)) if m else None

def extract_city(val):
    if pd.isna(val):
        return None
    return str(val).split(",")[0].strip()

df["storage_gb"] = df["storage"].apply(parse_storage)
df["city"] = df["location"].apply(extract_city)

features = ["brand", "model", "storage_gb", "condition_clean", "city", "photo_count", "seller_type"]
target = "price"

model_df = df.dropna(subset=["price", "brand", "storage_gb"]).copy()
print(f"usable rows after dropping missing price/brand/storage: {len(model_df)} / {len(df)}")

model_df[features].head()

## Step 3 — Prepare features and split into train/test

`brand`, `model`, `condition_clean`, `city`, `seller_type` are converted to pandas' `category` dtype so XGBoost can use its native categorical support — this handles `model`'s 208 distinct values without needing one-hot encoding (which would blow up into 208 separate columns, most of them nearly always zero).

**Price is log-transformed for training.** Prices in this dataset span ৳15,000 to ৳1,320,000 (an 88x range, median ৳39,500 but mean ৳53,538 — a few expensive phones pull the average way up). Training on raw price lets those high-value outliers dominate the error (RMSE squares errors, so being 15% off on a ৳1M phone counts far more than being 15% off on a ৳30k phone) — training on `log(price)` instead makes the model optimize for *relative* (percentage) accuracy across the whole range, which is what actually matters for a fair-price estimator. Predictions get converted back to real taka (`np.exp(...)`) everywhere you actually look at a number.

80% of the data trains the model; the other 20% is held back to honestly check how good it is on phones it's never seen.

In [ ]:
categorical_cols = ["brand", "model", "condition_clean", "city", "seller_type"]
numeric_cols = ["storage_gb", "photo_count"]

X = model_df[categorical_cols + numeric_cols].copy()
for col in categorical_cols:
    X[col] = X[col].astype("category")

y = model_df[target]
y_log = np.log(y)

X_train, X_test, y_train_log, y_test_log, y_train, y_test = train_test_split(
    X, y_log, y, test_size=0.2, random_state=42
)
print(f"train: {len(X_train)} rows | test: {len(X_test)} rows")

## Step 4 — Train the model

Trained on `y_train_log`, not raw price.

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    enable_categorical=True,
    random_state=42,
)
model.fit(X_train, y_train_log)
print("trained.")

## Step 5 — Evaluate: how far off is it, honestly?

The model predicts in log-space, so predictions are converted back with `np.exp(...)` before comparing to real prices. RMSE is in ৳ ("on average, how many taka was the model wrong by"); MAPE is in % ("on average, how far off, relatively") — MAPE is the more meaningful number here given how wide the price range is.

In [ ]:
y_pred_test_log = model.predict(X_test)
y_pred_test = np.exp(y_pred_test_log)

rmse = root_mean_squared_error(y_test, y_pred_test)
mape = mean_absolute_percentage_error(y_test, y_pred_test) * 100

print(f"RMSE on held-out test set: \u09f3{rmse:,.0f}")
print(f"MAPE on held-out test set: {mape:.1f}%")
print(f"(for reference, average price in the dataset is \u09f3{y.mean():,.0f}, median \u09f3{y.median():,.0f})")

## Step 6 — Fair price range + deal score, for every listing

For each listing: predict its fair price (converted back out of log-space), build a ±10% range around it, and compute a deal score comparing the *actual* asking price to that prediction.

`deal_score = 100 - ((actual_price - predicted_price) / predicted_price * 100)`, clipped to 0-100. A listing priced well below its predicted fair value scores high (good deal); priced above scores low.

In [ ]:
model_df["predicted_price"] = np.exp(model.predict(X))
model_df["fair_price_min"] = (model_df["predicted_price"] * 0.9).round(0)
model_df["fair_price_max"] = (model_df["predicted_price"] * 1.1).round(0)

raw_score = 100 - ((model_df["price"] - model_df["predicted_price"]) / model_df["predicted_price"] * 100)
model_df["deal_score"] = raw_score.clip(lower=0, upper=100).round(0)

model_df[["brand", "model", "price", "predicted_price", "fair_price_min", "fair_price_max", "deal_score"]].sort_values("deal_score", ascending=False).head(15)

## Step 7 — Save the model and results

The trained model gets saved so it can be reused later (e.g. by an API) without retraining. The full results table gets saved as CSV so you can inspect it or load it elsewhere.

Note: the saved model predicts **log(price)** — anything loading `fair_price_model.json` later needs to `np.exp(...)` its predictions too, same as this notebook does.

In [ ]:
model.save_model("fair_price_model.json")
model_df.to_csv("model_results.csv", index=False)
print("saved: ml/fair_price_model.json, ml/model_results.csv")